# ML-09 — Validation and Research Claim Audit

## Purpose

This notebook audits the Week-5 content refresh model from a validation, leakage, and claim-discipline perspective.

The audit has four goals:

1. Review two findings from the FlyRank research paper and ask constructive methodology questions.
2. Compare the Week-5 model under a naive random split and an honest client-grouped split.
3. Audit the final feature set for possible leakage.
4. Rewrite my strongest model claim so that it matches the evidence.

The intended use of this model is decision-support: ranking pages for human review rather than guaranteeing that a page will recover after a refresh.

All examples and outputs are kept public-safe. No client names, domains, URLs, private queries, or raw text are displayed.

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42

print("Setup complete.")

Setup complete.


## 1. Two paper findings + my methodology questions

The purpose of this section is not to grade the FlyRank research paper. I am using it as an example of how to review a result carefully.

For each finding, I ask two questions:

1. Where does the outcome label come from, and does it represent the real decision we care about?
2. Does the validation design support the generalization implied by the finding?

I am keeping the tone constructive and focusing on questions that I would also want a reviewer to ask about my own model.

### Finding 1 — The learned model improved Precision@50 over the rule-based baseline

One finding I would examine is that the Random Forest achieved a substantially higher Precision@50 than the rule-based baseline.

My methodology question would be:

**Where exactly does the positive label come from, and does that label represent the real-world outcome that the ranking system is intended to support?**

The model's positive class is based on the chosen decline label rather than a directly observed business outcome such as a confirmed successful content refresh.

I would therefore want to confirm that the label is defined using information available after the feature window and that the label window does not overlap the feature window.

A second question is whether the validation design matches the intended use. If pages from the same client can appear in both training and test data, the model may learn client-specific patterns rather than generalizable page-level signal.

I find the finding useful as decision-support evidence, but I would avoid interpreting the Precision@50 result as proof that the model will cause better content outcomes.

### Finding 2 — Search and content signals can support prioritization

A second finding I would examine is the broader conclusion that observable search and content signals can help identify pages that deserve review.

My methodology question would be:

**Does the validation design support a claim about prioritization only, or does it support a stronger claim about future business outcomes?**

The label is based on an observed outcome/proxy rather than an experimental intervention.

Therefore, a strong model score can show that the model is useful for ranking pages according to the selected label, but it cannot by itself show that refreshing those pages will cause traffic or visibility to recover.

I would also ask whether the validation set represents genuinely unseen clients or future observations. A random split could place highly related pages from the same client in both training and test data, making the evaluation easier than the real decision environment.

I would therefore interpret the finding as directional decision-support evidence rather than causal evidence.

In [2]:
# Try the common locations used in the previous notebooks.

candidate_paths = [
    "../data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
    "/content/data/raw/content_refresh_anonymized.csv",
    "/content/content_refresh_anonymized.csv"
]

data_path = None

for path in candidate_paths:
    if os.path.exists(path):
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError(
        "Could not find content_refresh_anonymized.csv. "
        "Check that the notebook is inside work/notebooks/ and the dataset is in data/raw/."
    )

df = pd.read_csv(data_path)

print("Dataset path:", data_path)
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

Dataset path: ../../data/raw/content_refresh_anonymized.csv
Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,NaN,NaN,11751,58,87,78,75,1,0,3,88,51,3626,22,35,4206,17,26,463,365+,6,22,0-30,NaN,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,NaN,gemini-3-flash-preview,19140,24,177,145,144,0,0,43,88,33,4211,10,14,6452,2,9,263,181-365,5,14,0-30,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [3]:
print("Columns:")
for col in df.columns:
    print("-", col)

Columns:
- content_id
- client_id
- search_volume
- competition
- competition_level
- cpc
- content_type
- main_intent
- word_count
- char_count
- provider_used
- model_used
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- days_with_impressions
- days_with_sessions
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- content_age_days
- age_tier
- age_tier_order
- days_since_last_update
- freshness_tier
- word_count_tier
- char_count_tier
- ctr
- avg_position
- engagement_rate
- scroll_rate
- ai_traffic_pct
- impression_tier
- position_tier
- trend_direction
- trend_pct


In [4]:
# Create the Week-5 target if the source trend column exists.

if "trend_direction" in df.columns:
    df["is_declining_label"] = (
        df["trend_direction"]
        .astype(str)
        .str.lower()
        .eq("down")
        .astype(int)
    )

elif "is_declining_label" not in df.columns:
    raise KeyError(
        "Neither 'trend_direction' nor 'is_declining_label' was found."
    )

# Identify client grouping column.
possible_group_cols = [
    "client_hash_id",
    "client_id",
    "client_hash",
    "client"
]

group_col = next(
    (c for c in possible_group_cols if c in df.columns),
    None
)

if group_col is None:
    raise KeyError(
        "Could not find a client grouping column. "
        "Expected something such as client_hash_id."
    )

print("Target:", "is_declining_label")
print("Group column:", group_col)

print("\nTarget distribution:")
display(df["is_declining_label"].value_counts(dropna=False))

print("\nNumber of clients:", df[group_col].nunique())

Target: is_declining_label
Group column: client_id

Target distribution:


is_declining_label
1    16262
0    13738
Name: count, dtype: int64


Number of clients: 32


## 2. My model under an honest split (before/after)

The Week-5 model was evaluated using a Random Forest and Precision@50 as the primary decision metric.

For this audit, I deliberately compare two validation designs:

**Before — naive random split**

Rows are randomly divided into training and test sets. This is easy to implement, but pages from the same client can appear in both sets.

**After — client-grouped split**

Entire clients are kept together so that a client is either in training or test, but not both.

The grouped split is a stricter test of generalization because the model must rank pages for clients whose examples were not available during training.

I will treat the grouped result as the more defensible estimate for this use case.

In [5]:
# Fields that should not be model features.

excluded_keywords = [
    "label",
    "target",
    "client_id",
    "client_hash",
    "content_id",
    "content_hash",
    "url",
    "query",
    "keyword",
    "title",
    "domain",
    "action_type",
    "priority_score",
    "health_score",
    "refresh_tier",
    "refresh_score"
]

excluded_columns = []

for col in df.columns:
    col_lower = col.lower()

    if col == "is_declining_label":
        excluded_columns.append(col)
        continue

    if any(keyword in col_lower for keyword in excluded_keywords):
        excluded_columns.append(col)

feature_cols = [
    col for col in df.columns
    if col not in excluded_columns
]

print("Excluded columns:")
print(excluded_columns)

print("\nCandidate model features:")
print(feature_cols)

Excluded columns:
['content_id', 'client_id', 'is_declining_label']

Candidate model features:
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [6]:
target_derived_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

feature_cols = [
    col for col in feature_cols
    if col not in target_derived_columns
]

print("Final feature columns:")
for col in feature_cols:
    print("-", col)

print("\nNumber of features:", len(feature_cols))

Final feature columns:
- search_volume
- competition
- competition_level
- cpc
- content_type
- main_intent
- word_count
- char_count
- provider_used
- model_used
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- days_with_impressions
- days_with_sessions
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- content_age_days
- age_tier
- age_tier_order
- days_since_last_update
- freshness_tier
- word_count_tier
- char_count_tier
- ctr
- avg_position
- engagement_rate
- scroll_rate
- ai_traffic_pct
- impression_tier
- position_tier

Number of features: 40


In [7]:
X = df[feature_cols].copy()
y = df["is_declining_label"].copy()
groups = df[group_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("groups:", groups.nunique())

X shape: (30000, 40)
y shape: (30000,)
groups: 32


In [8]:
numeric_features = X.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["number", "bool"]
).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
Categorical features: ['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']


In [9]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

print("Pipeline created.")

Pipeline created.


In [10]:
def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(scores))

    top_indices = np.argsort(scores)[::-1][:k]

    return y_true[top_indices].mean()

In [11]:
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

pipeline_random = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]
)

pipeline_random.fit(X_train_random, y_train_random)

random_scores = pipeline_random.predict_proba(X_test_random)[:, 1]

random_roc_auc = roc_auc_score(
    y_test_random,
    random_scores
)

random_ap = average_precision_score(
    y_test_random,
    random_scores
)

random_p50 = precision_at_k(
    y_test_random,
    random_scores,
    k=50
)

print("NAIVE RANDOM SPLIT")
print("------------------")
print(f"ROC-AUC:       {random_roc_auc:.3f}")
print(f"Average Prec.: {random_ap:.3f}")
print(f"Precision@50:  {random_p50:.3f}")

NAIVE RANDOM SPLIT
------------------
ROC-AUC:       0.893
Average Prec.: 0.907
Precision@50:  1.000


In [12]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_grouped = X.iloc[train_idx]
X_test_grouped = X.iloc[test_idx]

y_train_grouped = y.iloc[train_idx]
y_test_grouped = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Training rows:", len(X_train_grouped))
print("Test rows:", len(X_test_grouped))

print("Training clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

print(
    "Client overlap:",
    len(set(groups_train) & set(groups_test))
)

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0


In [13]:
pipeline_grouped = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]
)

pipeline_grouped.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_scores = pipeline_grouped.predict_proba(
    X_test_grouped
)[:, 1]

grouped_roc_auc = roc_auc_score(
    y_test_grouped,
    grouped_scores
)

grouped_ap = average_precision_score(
    y_test_grouped,
    grouped_scores
)

grouped_p50 = precision_at_k(
    y_test_grouped,
    grouped_scores,
    k=50
)

print("CLIENT-GROUPED SPLIT")
print("--------------------")
print(f"ROC-AUC:       {grouped_roc_auc:.3f}")
print(f"Average Prec.: {grouped_ap:.3f}")
print(f"Precision@50:  {grouped_p50:.3f}")

CLIENT-GROUPED SPLIT
--------------------
ROC-AUC:       0.827
Average Prec.: 0.821
Precision@50:  0.920


In [14]:
comparison = pd.DataFrame({
    "Validation": [
        "Naive random split",
        "Client-grouped split"
    ],
    "ROC-AUC": [
        random_roc_auc,
        grouped_roc_auc
    ],
    "Average Precision": [
        random_ap,
        grouped_ap
    ],
    "Precision@50": [
        random_p50,
        grouped_p50
    ]
})

display(
    comparison.style.format({
        "ROC-AUC": "{:.3f}",
        "Average Precision": "{:.3f}",
        "Precision@50": "{:.3f}"
    })
)

,Validation,ROC-AUC,Average Precision,Precision@50
0,Naive random split,0.893,0.907,1.000
1,Client-grouped split,0.827,0.821,0.920


In [15]:
gap_p50 = random_p50 - grouped_p50
gap_auc = random_roc_auc - grouped_roc_auc
gap_ap = random_ap - grouped_ap

print(f"Precision@50 gap: {gap_p50:.3f}")
print(f"ROC-AUC gap:      {gap_auc:.3f}")
print(f"Average precision gap: {gap_ap:.3f}")

Precision@50 gap: 0.080
ROC-AUC gap:      0.066
Average precision gap: 0.086


### Before/after interpretation

The naive random split produced a Precision@50 of **[RANDOM_P50]**, while the client-grouped split produced **[GROUPED_P50]**.

The difference is **[GAP]** Precision@50 points.

The grouped result is the more defensible number for this audit because the test clients were not present in the training data.

The comparison suggests that the naive random split may have benefited from client-specific patterns shared across training and test rows.

Therefore, I will not use the random-split score as my main evidence of generalization.

The client-grouped result is the number I would use for a public-safe, decision-support claim.

In [16]:
assert len(set(groups_train) & set(groups_test)) == 0

print("PASS: No client appears in both training and grouped test sets.")

PASS: No client appears in both training and grouped test sets.


## 3. Leakage audit

I am auditing the final feature set rather than only checking whether the model runs.

For every feature, I ask:

1. Could this field contain the answer directly?
2. Could this field have been calculated using future information?
3. Is it a product decision or score rather than an observable input?
4. Could it identify the client, URL, query, or content item and allow memorization?
5. Is it derived from the target?

The goal is to ensure that the model learns from information that would realistically be available at prediction time.

In [17]:
leakage_keywords = [
    "label",
    "target",
    "future",
    "outcome",
    "next",
    "action",
    "priority",
    "health",
    "refresh_score",
    "refresh_tier",
    "trend_direction",
    "trend_pct",
    "url",
    "query",
    "keyword",
    "domain",
    "client_id",
    "client_hash",
    "content_id",
    "content_hash"
]

leakage_candidates = []

for col in feature_cols:
    col_lower = col.lower()

    matched = [
        keyword
        for keyword in leakage_keywords
        if keyword in col_lower
    ]

    if matched:
        leakage_candidates.append({
            "feature": col,
            "matched_terms": ", ".join(matched)
        })

leakage_report = pd.DataFrame(leakage_candidates)

if len(leakage_report) == 0:
    print("No obvious leakage keywords found in the final feature set.")
else:
    print("Features requiring manual review:")
    display(leakage_report)

No obvious leakage keywords found in the final feature set.


In [18]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [19]:
if "content_hash_id" in df.columns:
    print(
        "Duplicate content IDs:",
        df["content_hash_id"].duplicated().sum()
    )

if group_col in df.columns:
    print(
        "Unique clients:",
        df[group_col].nunique()
    )

Unique clients: 32


In [20]:
missingness = (
    df[feature_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

missingness.columns = ["feature", "missing_rate"]

display(
    missingness.style.format({
        "missing_rate": "{:.1%}"
    })
)

,feature,missing_rate
0,provider_used,71.5%
1,word_count,25.7%
2,char_count_tier,25.7%
3,char_count,25.7%
4,word_count_tier,25.7%
5,model_used,19.1%
6,competition_level,8.7%
7,cpc,8.2%
8,competition,8.2%
9,search_volume,8.2%


In [21]:
audit_rows = []

for col in feature_cols:
    col_lower = col.lower()

    reasons = []

    if any(k in col_lower for k in [
        "label", "target", "future", "outcome"
    ]):
        reasons.append("possible target/future information")

    if any(k in col_lower for k in [
        "priority", "health", "action", "refresh_score"
    ]):
        reasons.append("possible product decision")

    if any(k in col_lower for k in [
        "url", "query", "keyword", "domain"
    ]):
        reasons.append("potentially identifying field")

    if "trend_direction" in col_lower:
        reasons.append("used to create target")

    audit_rows.append({
        "feature": col,
        "dtype": str(df[col].dtype),
        "missing_rate": df[col].isna().mean(),
        "review_flag": "; ".join(reasons) if reasons else "no obvious flag"
    })

feature_audit = pd.DataFrame(audit_rows)

display(feature_audit)

,feature,dtype,missing_rate,review_flag
0,search_volume,float64,0.082267,no obvious flag
1,competition,float64,0.082267,no obvious flag
2,competition_level,object,0.087000,no obvious flag
3,cpc,float64,0.082267,no obvious flag
4,content_type,object,0.000000,no obvious flag
5,main_intent,object,0.079133,no obvious flag
6,word_count,float64,0.256633,no obvious flag
7,char_count,float64,0.256633,no obvious flag
8,provider_used,object,0.714600,no obvious flag
9,model_used,object,0.191100,no obvious flag


### Leakage audit conclusion

The final feature set was reviewed for target-derived fields, future information, product decision outputs, identifying fields, and duplicated observations.

The most important exclusion is the target-derived trend information. Because the label is defined from the decline direction, fields that directly encode that direction must not be used as model features.

I also excluded client/content identifiers from the feature matrix. These fields can be useful for grouping and audit work but should not be treated as predictive signals.

The remaining features are treated as observable signals available before the prediction decision.

I therefore consider the final feature set substantially safer than a feature set that includes target-derived or product-decision fields. This is a methodological conclusion from the audit, not proof that every possible leakage path has been eliminated.

### Real failure examples

A validation metric summarizes performance, but it does not explain why individual predictions are wrong.

I therefore inspect examples from the grouped test set.

For public safety, I will only show pseudonymized identifiers and model/label information. I will not display URLs, client names, search queries, titles, or raw content.

In [22]:
failure_df = X_test_grouped.copy()

failure_df["actual"] = y_test_grouped.values
failure_df["predicted_probability"] = grouped_scores

failure_df["predicted_class"] = (
    failure_df["predicted_probability"] >= 0.50
).astype(int)

failure_df["error_type"] = "correct"

failure_df.loc[
    (failure_df["actual"] == 0) &
    (failure_df["predicted_class"] == 1),
    "error_type"
] = "false_positive"

failure_df.loc[
    (failure_df["actual"] == 1) &
    (failure_df["predicted_class"] == 0),
    "error_type"
] = "false_negative"

failure_df["absolute_error"] = abs(
    failure_df["actual"] -
    failure_df["predicted_probability"]
)

failure_df.head()

,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,actual,predicted_probability,predicted_class,error_type,absolute_error
0,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,1,0.700146,1,correct,0.299854
1,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,1,0.476540,0,false_negative,0.523460
5,720.0,1.00,HIGH,1.05,keyword article,transactional,3080.0,18178.0,NaN,gemini-2.5-flash,3970,1,4,5,5,0,0,1,88,5,617,0,4,1009,1,1,147,91-180,4,20,0-30,2000-3500,15000-25000,0.03,8.5,0.00,25.00,0.0,good,page_1,1,0.744490,1,correct,0.255510
13,10.0,0.00,LOW,0.00,keyword article,informational,1342.0,8469.0,NaN,gpt-4o-mini,307,0,4,4,4,0,0,1,69,3,85,0,3,77,0,1,238,181-365,5,103,91-180,1000-2000,8000-15000,0.00,39.8,0.00,25.00,0.0,moderate,page_3_5,0,0.774344,1,false_positive,0.774344
19,30.0,1.00,HIGH,1.57,keyword article,transactional,2673.0,16168.0,NaN,gemini-2.5-flash,99,2,2,3,3,0,0,1,56,3,10,0,2,31,0,0,187,181-365,5,20,0-30,2000-3500,15000-25000,2.02,6.9,0.00,50.00,0.0,low,page_1,1,0.580293,1,correct,0.419707


In [23]:
false_positives = (
    failure_df[
        failure_df["error_type"] == "false_positive"
    ]
    .sort_values(
        "predicted_probability",
        ascending=False
    )
)

print("False positives:", len(false_positives))

display(
    false_positives[
        ["actual", "predicted_probability", "predicted_class"]
    ].head(10)
)

False positives: 1004


,actual,predicted_probability,predicted_class
20736,0,0.852667,1
2357,0,0.849650,1
12332,0,0.842255,1
11061,0,0.835199,1
29456,0,0.823448,1
28718,0,0.821699,1
5399,0,0.817842,1
4050,0,0.812303,1
1517,0,0.811803,1
10080,0,0.807136,1


In [24]:
false_negatives = (
    failure_df[
        failure_df["error_type"] == "false_negative"
    ]
    .sort_values(
        "predicted_probability",
        ascending=False
    )
)

print("False negatives:", len(false_negatives))

display(
    false_negatives[
        ["actual", "predicted_probability", "predicted_class"]
    ].head(10)
)

False negatives: 665


,actual,predicted_probability,predicted_class
18345,1,0.499737,0
2121,1,0.499510,0
26986,1,0.498842,0
4854,1,0.498784,0
27043,1,0.498689,0
19714,1,0.498622,0
29688,1,0.498479,0
27242,1,0.498395,0
8388,1,0.498105,0
132,1,0.497933,0


### Failure interpretation

The false-positive examples are pages that received relatively high predicted decline probability but did not receive the positive decline label.

The false-negative examples are pages that received lower predicted probability but were positive according to the evaluation label.

These examples show why the model should be used as a ranking and review aid rather than an automatic decision-maker.

A prediction error does not necessarily mean the model is defective. It can also indicate that the selected proxy label does not fully capture the underlying content situation.

The examples reinforce the need for human review before taking a content action.

## 4. Claim rewrite

The original Week-5 result can easily be described too strongly.

A strong claim such as:

> "The Random Forest predicts which pages will decline."

goes beyond what the validation and label design establish.

The model was evaluated against a proxy label and is intended to support prioritization.

I therefore rewrite the claim using safe language: observed, measured, directional, and decision-support.

### Original claim

> The Random Forest predicts which pages will decline.

### Safer claim

> In this anonymized evaluation, the Random Forest **measured higher ranking performance than the rule-based baseline on the selected decline proxy**, and the client-grouped validation provides a more conservative estimate of performance on unseen clients. The result is **directional decision-support evidence** for prioritizing pages for human review; it does not establish that the model will cause traffic recovery or that a refresh will produce a particular outcome.

### Why I changed it

The original statement implies a stronger predictive and business guarantee than the evidence supports.

The revised statement:

- identifies the evaluation setting;
- refers to the selected proxy label;
- distinguishes measurement from causality;
- uses "directional" language;
- frames the model as decision-support;
- avoids claiming that a content refresh will cause recovery.

In [25]:
print("Final grouped validation metrics")
print("--------------------------------")
print(f"ROC-AUC: {grouped_roc_auc:.3f}")
print(f"Average Precision: {grouped_ap:.3f}")
print(f"Precision@50: {grouped_p50:.3f}")

Final grouped validation metrics
--------------------------------
ROC-AUC: 0.827
Average Precision: 0.821
Precision@50: 0.920


### Measured result

Under the client-grouped validation split, the model achieved:

- ROC-AUC: **[YOUR VALUE]**
- Average Precision: **[YOUR VALUE]**
- Precision@50: **[YOUR VALUE]**

Precision@50 is the primary metric because the practical use case is a ranked review queue: reviewers have limited capacity and need to inspect the highest-priority pages first.

These numbers are measured results on the evaluated dataset and should not be interpreted as guarantees of performance on future datasets, clients, or business outcomes.

## Self-check

Before submitting, I confirm:

- [ ] Every section is filled with both Markdown reasoning and supporting code.
- [ ] The notebook runs from top to bottom without errors.
- [ ] I compared a naive random split with a client-grouped split.
- [ ] I verified that the grouped train and test sets have zero client overlap.
- [ ] I used Precision@50 as the main decision-support metric.
- [ ] I included ROC-AUC and Average Precision as supporting metrics.
- [ ] I audited the final feature set for target leakage and future information.
- [ ] I did not use client names, domains, URLs, private queries, or raw text.
- [ ] I inspected real false-positive and false-negative examples.
- [ ] My claims use careful language such as observed, measured, directional, and decision-support.
- [ ] I did not claim that the model causes traffic recovery.
- [ ] I did not claim that the model proves anything about Google's algorithm.
- [ ] I saved the executed notebook.
- [ ] I committed `work/notebooks/w06_validation_audit.ipynb` to my repository.
- [ ] I will submit my repository URL on the assignment card.